# Operations RCA NLP: Aviation Full-Dataset EDA and Baseline Training

This notebook provides a clean workflow for:

1. Loading the full original aviation dataset files
2. Running safe early EDA summaries and charts
3. Training a TF-IDF + One-vs-Rest Logistic Regression baseline
4. Evaluating on internal split and official test split
5. Exporting model artifacts for local app use

Safety notes:

- Do not print raw narrative rows.
- Do not commit raw data.
- Do not commit `model.joblib`.

## 1. Imports

In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import date
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, hamming_loss

## 2. Project Paths and Configuration

In [ ]:
ROOT = Path.cwd().resolve().parent
RAW_DIR = ROOT / "data" / "raw"
ARTIFACT_DIR = ROOT / "artifacts" / "aviation"
TMP_DIR = Path("/tmp")

TRAIN_TEXT_PATH = RAW_DIR / "TrainingData.txt"
TRAIN_LABELS_PATH = RAW_DIR / "TrainCategoryMatrix.csv"
TEST_TEXT_PATH = RAW_DIR / "TestData.txt"
TEST_LABELS_PATH = RAW_DIR / "TestTruth.csv"

LABEL_COLUMNS = [f"Anomaly_{i}" for i in range(1, 23)]
THRESHOLD = 0.50

TFIDF_SETTINGS = {
    "ngram_range": (1, 2),
    "max_features": 30000,
    "min_df": 2,
    "strip_accents": "unicode",
    "sublinear_tf": True,
}

LOGREG_SETTINGS = {
    "C": 2.0,
    "max_iter": 400,
    "class_weight": "balanced",
    "solver": "liblinear",
    "random_state": 42,
}

print("ROOT:", ROOT)
print("RAW_DIR:", RAW_DIR)
print("ARTIFACT_DIR:", ARTIFACT_DIR)

## 3. Dataset File Presence Check

In [ ]:
required_files = [
    TRAIN_TEXT_PATH,
    TRAIN_LABELS_PATH,
    TEST_TEXT_PATH,
    TEST_LABELS_PATH,
]

missing = [p for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

for p in required_files:
    print(f"{p.name}: exists=True size_bytes={p.stat().st_size}")

## 4. Helper Functions (Safe Loading and Validation)

In [ ]:
def load_text_lines(path: Path) -> pd.Series:
    """Load text file and drop blank separator lines."""
    lines = path.read_text(encoding="utf-8").splitlines()
    clean = [line.strip() for line in lines if line.strip()]
    return pd.Series(clean, name="text", dtype="string")


def load_label_matrix(path: Path, label_columns: list[str]) -> pd.DataFrame:
    """Load label matrix and map {-1,1} to {0,1}."""
    df = pd.read_csv(path, header=None)
    if df.shape[1] != len(label_columns):
        raise ValueError(
            f"Expected {len(label_columns)} label columns, got {df.shape[1]}"
        )
    df.columns = label_columns
    df = df.replace({-1: 0, 1: 1}).astype(int)
    return df


def load_split_dataset(
    text_path: Path,
    labels_path: Path,
    label_columns: list[str],
) -> tuple[pd.Series, pd.DataFrame]:
    texts = load_text_lines(text_path).reset_index(drop=True)
    labels = load_label_matrix(labels_path, label_columns).reset_index(drop=True)
    if len(texts) != len(labels):
        raise ValueError(
            f"Row mismatch: text_rows={len(texts)} label_rows={len(labels)}"
        )
    return texts, labels


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "micro_f1": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "samples_f1": float(f1_score(y_true, y_pred, average="samples", zero_division=0)),
        "hamming_loss": float(hamming_loss(y_true, y_pred)),
    }

## 5. Load Train/Test Splits from Original Dataset

In [ ]:
train_texts, train_labels = load_split_dataset(
    TRAIN_TEXT_PATH,
    TRAIN_LABELS_PATH,
    LABEL_COLUMNS,
)
test_texts, test_labels = load_split_dataset(
    TEST_TEXT_PATH,
    TEST_LABELS_PATH,
    LABEL_COLUMNS,
)

print("train_rows:", len(train_texts))
print("test_rows:", len(test_texts))
print("label_count:", train_labels.shape[1])

## 6. Safe Early EDA Summary

In [ ]:
eda_summary = pd.DataFrame(
    {
        "metric": [
            "train_rows",
            "test_rows",
            "label_count",
            "train_missing_text_values",
            "test_missing_text_values",
            "train_missing_label_values",
            "test_missing_label_values",
        ],
        "value": [
            len(train_texts),
            len(test_texts),
            len(LABEL_COLUMNS),
            int(train_texts.isna().sum()),
            int(test_texts.isna().sum()),
            int(train_labels.isna().sum().sum()),
            int(test_labels.isna().sum().sum()),
        ],
    }
)
eda_summary

In [ ]:
label_prevalence_train = train_labels.mean().sort_values(ascending=False)
label_prevalence_test = test_labels.mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 4), constrained_layout=True)
label_prevalence_train.plot(kind="bar", ax=axes[0], title="Train Label Prevalence")
label_prevalence_test.plot(kind="bar", ax=axes[1], title="Test Label Prevalence")
axes[0].set_ylabel("positive_rate")
axes[1].set_ylabel("positive_rate")
plt.show()

In [ ]:
train_len = train_texts.str.len()
test_len = test_texts.str.len()

length_summary = pd.DataFrame(
    {
        "split": ["train", "test"],
        "min_chars": [int(train_len.min()), int(test_len.min())],
        "p50_chars": [int(train_len.quantile(0.50)), int(test_len.quantile(0.50))],
        "p90_chars": [int(train_len.quantile(0.90)), int(test_len.quantile(0.90))],
        "max_chars": [int(train_len.max()), int(test_len.max())],
    }
)
length_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
axes[0].hist(train_len, bins=40)
axes[0].set_title("Train Narrative Length Distribution")
axes[0].set_xlabel("characters")

axes[1].hist(test_len, bins=40)
axes[1].set_title("Test Narrative Length Distribution")
axes[1].set_xlabel("characters")
plt.show()

## 7. Internal Validation Split Training

In [ ]:
x_train, x_valid, y_train, y_valid = train_test_split(
    train_texts,
    train_labels,
    test_size=0.20,
    random_state=42,
)

vectorizer_valid = TfidfVectorizer(**TFIDF_SETTINGS)
x_train_vec = vectorizer_valid.fit_transform(x_train)
x_valid_vec = vectorizer_valid.transform(x_valid)

clf_valid = OneVsRestClassifier(LogisticRegression(**LOGREG_SETTINGS))
clf_valid.fit(x_train_vec, y_train)

valid_proba = clf_valid.predict_proba(x_valid_vec)
valid_pred = (np.asarray(valid_proba) >= THRESHOLD).astype(int)

internal_metrics = compute_metrics(y_valid.to_numpy(), valid_pred)
internal_metrics

## 8. Official Test Split Evaluation

In [ ]:
x_test_vec = vectorizer_valid.transform(test_texts)
test_proba = clf_valid.predict_proba(x_test_vec)
test_pred = (np.asarray(test_proba) >= THRESHOLD).astype(int)

official_test_metrics = compute_metrics(test_labels.to_numpy(), test_pred)
official_test_metrics

## 9. Train Final Artifact Model on Full Training Split

In [ ]:
final_vectorizer = TfidfVectorizer(**TFIDF_SETTINGS)
x_full_vec = final_vectorizer.fit_transform(train_texts)

final_classifier = OneVsRestClassifier(LogisticRegression(**LOGREG_SETTINGS))
final_classifier.fit(x_full_vec, train_labels)

final_metadata = {
    "domain_id": "aviation",
    "domain": "aviation",
    "model_name": "TF-IDF + One-vs-Rest Logistic Regression",
    "model_type": "multi-label classification",
    "training_approach": "TF-IDF vectorization with One-vs-Rest Logistic Regression",
    "version": "aviation-full-v0.1.1",
    "training_date": str(date.today()),
    "threshold_default": float(THRESHOLD),
    "threshold": float(THRESHOLD),
    "label_count": len(LABEL_COLUMNS),
    "labels": LABEL_COLUMNS,
    "tfidf_settings": {
        "ngram_range": list(TFIDF_SETTINGS["ngram_range"]),
        "max_features": TFIDF_SETTINGS["max_features"],
        "min_df": TFIDF_SETTINGS["min_df"],
        "strip_accents": TFIDF_SETTINGS["strip_accents"],
        "sublinear_tf": TFIDF_SETTINGS["sublinear_tf"],
    },
    "logistic_regression_settings": LOGREG_SETTINGS,
    "evaluation_metrics": official_test_metrics,
    "metrics": official_test_metrics,
    "dataset_provenance_note": "Full original aviation dataset from local files (text + category matrices).",
    "limitation_note": "Outputs are likely root-cause-related factor indicators for analyst review support and do not establish definitive causality.",
    "responsible_use_note": "Decision support only; requires human analyst review and must not be used as a certified operational safety decision system.",
    "data_input_format_note": "split files (TrainingData.txt, TrainCategoryMatrix.csv)",
}

final_label_mapping = {
    label: {
        "display_name": label.replace("_", " "),
        "description": f"Aviation anomaly-factor label '{label}'. Contributory-factor indicator for analyst review.",
    }
    for label in LABEL_COLUMNS
}

final_payload = {
    "vectorizer": final_vectorizer,
    "classifier": final_classifier,
    "labels": LABEL_COLUMNS,
    "metadata": final_metadata,
}

## 10. Export Artifacts

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

model_path = ARTIFACT_DIR / "model.joblib"
metadata_path = ARTIFACT_DIR / "metadata.json"
label_mapping_path = ARTIFACT_DIR / "label_mapping.json"

joblib.dump(final_payload, model_path)
metadata_path.write_text(json.dumps(final_metadata, indent=2), encoding="utf-8")
label_mapping_path.write_text(json.dumps(final_label_mapping, indent=2), encoding="utf-8")

print("wrote:", model_path)
print("wrote:", metadata_path)
print("wrote:", label_mapping_path)

## 11. Save Temporary Combined CSVs (Optional)

Use this only if you want portable combined CSV files for other scripts.
These files are written to `/tmp` and are not part of the repository.

In [ ]:
train_combined = pd.concat([train_texts.rename("text"), train_labels], axis=1)
test_combined = pd.concat([test_texts.rename("text"), test_labels], axis=1)

tmp_train_csv = TMP_DIR / "aviation_full_training.csv"
tmp_test_csv = TMP_DIR / "aviation_full_test.csv"

train_combined.to_csv(tmp_train_csv, index=False)
test_combined.to_csv(tmp_test_csv, index=False)

print("wrote:", tmp_train_csv, "rows=", len(train_combined))
print("wrote:", tmp_test_csv, "rows=", len(test_combined))

## 12. Next Steps

1. Validate API behavior locally with `/model-info`, `/predict`, and `/predict-batch`.
2. Upload `artifacts/aviation/model.joblib` to external storage (for example Hugging Face).
3. Update Render `MODEL_ARTIFACT_URL` and redeploy.
4. Keep raw data and model binaries out of Git history.